In [1]:
import pandas as pd

def print_annotation_csv(path, name, n=10):
    print(f"\n📄 ANNOTATION FILE PREVIEW — {name} (first {n} rows)")
    ann = pd.read_csv(path)
    print(ann.head(n))

    print("\n📌 Annotation columns:")
    print(list(ann.columns))

    # Print unique cell types
    for col in ["cell_type", "cell_type_mmc_raw", "cell_label", "cell_id"]:
        if col in ann.columns:
            print(f"\n🔍 Unique values in '{col}' (up to 20):")
            print(ann[col].drop_duplicates().head(20).tolist())
    print("="*80)

ZHUANG_ANN_PATH = "/p/project1/hai_fzj_bda/salg1/cellseg-benchmark/data_dir/samples/zhuang/results/merfish/cell_type_annotation/adata_obs_annotated.csv"
ZENG_ANN_PATH   = "/p/project1/hai_fzj_bda/salg1/cellseg-benchmark/data_dir/samples/zeng/results/merfish/cell_type_annotation/adata_obs_annotated.csv"


print_annotation_csv(ZENG_ANN_PATH, "ZENG")
print_annotation_csv(ZHUANG_ANN_PATH, "ZHUANG")


📄 ANNOTATION FILE PREVIEW — ZENG (first 10 rows)


/tmp/ipykernel_663287/1478567476.py:5: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  ann = pd.read_csv(path)


   average_correlation_score              cell_id cell_type_mmc_raw  \
0                   0.502280  1019171911101460569      Neurons-Glut   
1                   0.565058  1019171911101550321      Neurons-Glut   
2                   0.570642  1019171911100841066      Neurons-Glut   
3                   0.597115  1019171911101400425     Neurons-Other   
4                   0.594055  1019171911101380264      Neurons-Glut   
5                   0.548506  1019171911101130445     Neurons-Other   
6                   0.567890  1019171911101280013     Neurons-Other   
7                   0.602693  1019171911101120478     Neurons-Other   
8                   0.643913  1019171911101130255     Neurons-Other   
9                   0.638337  1019171911101110358     Neurons-Other   

  cell_type_mmc_incl_low_quality cell_type_mmc_is_mixed  \
0                      Undefined                 unique   
1                   Neurons-Glut                 unique   
2                   Neurons-Glut         

In [4]:
import os
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------
# Helpers
# ---------------------------------------------------

def sanity_check(adata, color=None, name="adata"):
    print("\n======================")
    print(f"🔍   ANNDATA CHECK ({name})")
    print("======================")
    print(f"Shape: {adata.n_obs:,} cells × {adata.n_vars:,} genes")
    print("obs columns:", list(adata.obs.columns))
    print("obsm keys:", list(adata.obsm.keys()))

    if color is not None:
        if color in adata.obs:
            print(f"\nColumn '{color}': {adata.obs[color].nunique()} unique values")
            print(f"Missing: {adata.obs[color].isna().sum():,}")
        else:
            print(f"\n⚠️ Column '{color}' NOT FOUND in obs")

    if "X_umap" in adata.obsm:
        print("UMAP present:", adata.obsm["X_umap"].shape)
    else:
        print("No UMAP yet")
    print("======================\n")

# ---------------------------------------------------
# LOAD EMBEDDINGS + ANNOTATIONS
# ---------------------------------------------------

ZHUANG_PATH = "/p/project1/hai_fzj_bda/spitzer2/point_transformer/data/raw/Zhuang-ABCA-1_concept_embeddings.h5ad"
ZENG_PATH   = "/p/project1/hai_fzj_bda/spitzer2/point_transformer/data/raw/Zeng_concept_embeddings.h5ad"
ISD_PATH    = "/p/project1/hai_fzj_bda/spitzer2/point_transformer/data/processed/ISD-1_concept_embeddings.h5ad"


ZHUANG_ANN_PATH = "/p/project1/hai_fzj_bda/salg1/cellseg-benchmark/data_dir/samples/zhuang/results/merfish/cell_type_annotation/adata_obs_annotated.csv"
ZENG_ANN_PATH   = "/p/project1/hai_fzj_bda/salg1/cellseg-benchmark/data_dir/samples/zeng/results/merfish/cell_type_annotation/adata_obs_annotated.csv"


SUBSAMPLE = 100_000

In [5]:
# ---------------------------------------------------
# Load ZHUANG
# ---------------------------------------------------
print(f"📂 Loading Zhuang: {ZHUANG_PATH}")
zhuang = sc.read(ZHUANG_PATH)
zhuang.obs_names = zhuang.obs_names.astype(str)
zhuang.obs["cell_id_raw"] = zhuang.obs_names
zhuang.obs_names = zhuang.obs["cell_id_raw"] + "-Zhuang"
zhuang.obs["cell_id"] = zhuang.obs_names
sc.pp.subsample(zhuang, n_obs=SUBSAMPLE, random_state=0)

sanity_check(zhuang, name="Zhuang")
# ---------------------------------------------------
# Load ZENG
# ---------------------------------------------------
print(f"📂 Loading Zeng: {ZENG_PATH}")
zeng = sc.read(ZENG_PATH)
zeng.obs_names = zeng.obs_names.astype(str)
zeng.obs["cell_id_raw"] = zeng.obs_names
zeng.obs_names = zeng.obs["cell_id_raw"] + "-Zeng"
zeng.obs["cell_id"] = zeng.obs_names
sc.pp.subsample(zeng, n_obs=SUBSAMPLE, random_state=0)

sanity_check(zeng, name="Zeng")

📂 Loading Zhuang: /p/project1/hai_fzj_bda/spitzer2/point_transformer/data/raw/Zhuang-ABCA-1_concept_embeddings.h5ad

🔍   ANNDATA CHECK (Zhuang)
Shape: 100,000 cells × 1,122 genes
obs columns: ['abc_sample_id', 'brain_section_label', 'brain_section_label_adata', 'class', 'class_color', 'cluster', 'cluster_alias', 'cluster_color', 'cluster_confidence_score', 'donor_genotype', 'donor_label', 'donor_sex', 'feature_matrix_label', 'high_quality_transfer', 'neurotransmitter', 'neurotransmitter_color', 'parcellation_category', 'parcellation_category_color', 'parcellation_division', 'parcellation_division_color', 'parcellation_index', 'parcellation_organ', 'parcellation_organ_color', 'parcellation_structure', 'parcellation_structure_color', 'parcellation_substructure', 'parcellation_substructure_color', 'subclass', 'subclass_color', 'subclass_confidence_score', 'supertype', 'supertype_color', 'x', 'x_ccf', 'y', 'y_ccf', 'z', 'z_ccf', 'cell_id_raw', 'cell_id']
obsm keys: ['concept_cls_embedding'

In [6]:
print("\n🔍 Checking Zeng ID structure…")

# 1. Extract base ID inside .obs (correct way)
zeng.obs["base_id"] = zeng.obs["cell_id"].str.replace(
    r"-\d+(?=-Zeng$)", "", regex=True
)

# 2. Count how many had suffixes (-1, -2, …)
num_with_suffix = (zeng.obs["cell_id"] != zeng.obs["base_id"] + "-Zeng").sum()
print(f"Cells with '-1'/'-2' suffixes: {num_with_suffix:,}")

# 3. Check for duplicated base IDs (multiple fragments)
dups = zeng.obs["base_id"][zeng.obs["base_id"].duplicated(keep=False)]
print(f"Base IDs appearing more than once: {dups.nunique():,}")

# 4. Show examples
if len(dups) > 0:
    print("\n📌 Example multi-fragment IDs:")
    for base in dups.unique()[:10]:
        variants = zeng.obs["cell_id"][zeng.obs["base_id"] == base].tolist()
        print(f"{base}: {variants}")
else:
    print("\n✅ No fragment suffixes detected — SAFE, nothing to strip")


🔍 Checking Zeng ID structure…
Cells with '-1'/'-2' suffixes: 100,000
Base IDs appearing more than once: 1,423

📌 Example multi-fragment IDs:
1018093344101150510-Zeng: ['1018093344101150510-Zeng', '1018093344101150510-1-Zeng']
1104095349100570323-Zeng: ['1104095349100570323-Zeng', '1104095349100570323-1-Zeng']
1018093344102620322-Zeng: ['1018093344102620322-3-Zeng', '1018093344102620322-1-Zeng', '1018093344102620322-Zeng']
1018093344101980294-Zeng: ['1018093344101980294-4-Zeng', '1018093344101980294-1-Zeng']
1018093344101340319-Zeng: ['1018093344101340319-1-Zeng', '1018093344101340319-3-Zeng']
1018093344100780684-Zeng: ['1018093344100780684-1-Zeng', '1018093344100780684-Zeng']
1018093344101840266-Zeng: ['1018093344101840266-5-Zeng', '1018093344101840266-2-Zeng']
1018093344102160232-Zeng: ['1018093344102160232-4-Zeng', '1018093344102160232-2-Zeng']
1018093344102210368-Zeng: ['1018093344102210368-2-Zeng', '1018093344102210368-3-Zeng']
1018093344100970462-Zeng: ['1018093344100970462-5-Zen

In [7]:
def print_cell_ids(adata, name, n=10):
    print(f"\n📌 {name} — FIRST {n} OBS NAMES:")
    print(adata.obs_names[:n].tolist())

    if "cell_id" in adata.obs:
        print(f"📌 {name} — FIRST {n} cell_id:")
        print(adata.obs['cell_id'][:n].tolist())
    else:
        print("⚠️ No 'cell_id' column in this AnnData.")

    if "cell_type" in adata.obs:
        print(f"📌 {name} — FIRST {n} cell_type:")
        print(adata.obs['cell_type'][:n].tolist())
    else:
        print("⚠️ No 'cell_type' column in this AnnData.")

    print("\n🔍 Counts:")
    if "cell_type" in adata.obs:
        print(adata.obs['cell_type'].value_counts(dropna=False).head())
    print("="*60)
print_cell_ids(zhuang, "ZHUANG")
print_cell_ids(zeng, "ZENG")


📌 ZHUANG — FIRST 10 OBS NAMES:
['196269251661640657821558596194131516996-Zhuang', '74694989265068621136814017210917662188-Zhuang', '153481091143684790537764283973536365605-Zhuang', '35754669376327154389380727082032974607-Zhuang', '209284113510049681080388547425982874009-Zhuang', '297432916244828122182771714624565340927-Zhuang', '246821969892156900584746899748023911665-Zhuang', '195019211204716016282645476521166239247-Zhuang', '186322290372526148915740912556582003868-Zhuang', '224696648076438278786182882631045664437-Zhuang']
📌 ZHUANG — FIRST 10 cell_id:
['196269251661640657821558596194131516996-Zhuang', '74694989265068621136814017210917662188-Zhuang', '153481091143684790537764283973536365605-Zhuang', '35754669376327154389380727082032974607-Zhuang', '209284113510049681080388547425982874009-Zhuang', '297432916244828122182771714624565340927-Zhuang', '246821969892156900584746899748023911665-Zhuang', '195019211204716016282645476521166239247-Zhuang', '186322290372526148915740912556582003868-

In [8]:
# Load annotation
z_ann = pd.read_csv(ZHUANG_ANN_PATH)
z_ann["base_id"] = z_ann["cell_id"].astype(str).str.split("-").str[0]
z_ann["cell_id_combined"] = z_ann["base_id"] + "-Zhuang"
z_ann = z_ann.rename(columns={"cell_type_mmc_raw": "cell_type"})

print("\n📄 ZHUANG ANNOTATION (cell_id_combined, cell_type):")
print(z_ann[["cell_id_combined", "cell_type"]].head(20))
print(f"Total rows: {len(z_ann):,}")


# Load annotation
ze_ann = pd.read_csv(ZENG_ANN_PATH)
ze_ann["cell_id_combined"] = ze_ann["cell_id"].astype(str) + "-Zeng"
ze_ann = ze_ann.rename(columns={"cell_type_mmc_raw": "cell_type"})

print("\n📄 ZENG ANNOTATION (cell_id_combined, cell_type):")
print(ze_ann[["cell_id_combined", "cell_type"]].head(20))
print(f"Total rows: {len(ze_ann):,}")


📄 ZHUANG ANNOTATION (cell_id_combined, cell_type):
                                  cell_id_combined     cell_type
0    38542084966093731202691164843098659276-Zhuang  Neurons-Glut
1   183256764301191388575899679882178798208-Zhuang  Neurons-Glut
2    37353846072841168055577846846204745147-Zhuang  Neurons-Glut
3    56522635880108955114956785931951865125-Zhuang  Neurons-Glut
4   114075972918310011122352881571035240931-Zhuang  Neurons-Glut
5   161395906901535153611343134398807012103-Zhuang  Neurons-Glut
6   196430079481818150765904306976008332942-Zhuang  Neurons-Glut
7   301630835670385691349511479183791495268-Zhuang  Neurons-Glut
8   118531435515176328294928752438827327938-Zhuang  Neurons-Glut
9   187669921940761829522139776202588300515-Zhuang  Neurons-Glut
10  320963102701035973479619920837315703378-Zhuang  Neurons-Glut
11   69288500551118823610104206177484494301-Zhuang  Neurons-Glut
12   95054221102774891854992075933030591651-Zhuang  Neurons-Glut
13   321618764101676563422956261780591

/tmp/ipykernel_663287/3244544377.py:13: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  ze_ann = pd.read_csv(ZENG_ANN_PATH)



📄 ZENG ANNOTATION (cell_id_combined, cell_type):
            cell_id_combined      cell_type
0   1019171911101460569-Zeng   Neurons-Glut
1   1019171911101550321-Zeng   Neurons-Glut
2   1019171911100841066-Zeng   Neurons-Glut
3   1019171911101400425-Zeng  Neurons-Other
4   1019171911101380264-Zeng   Neurons-Glut
5   1019171911101130445-Zeng  Neurons-Other
6   1019171911101280013-Zeng  Neurons-Other
7   1019171911101120478-Zeng  Neurons-Other
8   1019171911101130255-Zeng  Neurons-Other
9   1019171911101110358-Zeng  Neurons-Other
10  1019171911101540031-Zeng   Neurons-Glut
11  1019171911101450426-Zeng   Neurons-Glut
12  1019171911101130343-Zeng  Neurons-Other
13  1019171911101120455-Zeng  Neurons-Other
14  1019171911101120292-Zeng  Neurons-Other
15  1019171911101130410-Zeng  Neurons-Other
16  1019171911101130177-Zeng  Neurons-Other
17  1019171911101120409-Zeng  Neurons-Other
18  1019171911101110571-Zeng  Neurons-Other
19  1019171911101270045-Zeng  Neurons-Other
Total rows: 3,739,961
